# HW 1- Word2Vec
BUDT 758O:  Special Topics in Decision, Operations and Information Technologies; Designing AI Systems

---

**Shruti Elango (selango4)**

Data: WikiText-2 and WikiText103


### Processing the Data





In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import random
import nltk
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Wikitext-2
wikitext_2_valid = pd.read_parquet('/content/drive/MyDrive/BUDT 758O:  Special Topics in Decision, Operations and Information Technologies; Designing AI Systems/HW1/wikitext-2/validation-2.parquet')
wikitext_2_test = pd.read_parquet('/content/drive/MyDrive/BUDT 758O:  Special Topics in Decision, Operations and Information Technologies; Designing AI Systems/HW1/wikitext-2/test-2.parquet')
wikitext_2_train = pd.read_parquet('/content/drive/MyDrive/BUDT 758O:  Special Topics in Decision, Operations and Information Technologies; Designing AI Systems/HW1/wikitext-2/train-2.parquet')
#Wikitext-103
wikitext_103_valid = pd.read_parquet('/content/drive/MyDrive/BUDT 758O:  Special Topics in Decision, Operations and Information Technologies; Designing AI Systems/HW1/wikitext-103/validation-103.parquet')
wikitext_103_test = pd.read_parquet('/content/drive/MyDrive/BUDT 758O:  Special Topics in Decision, Operations and Information Technologies; Designing AI Systems/HW1/wikitext-103/test-103.parquet')
wikitext_103_train_1 = pd.read_parquet('/content/drive/MyDrive/BUDT 758O:  Special Topics in Decision, Operations and Information Technologies; Designing AI Systems/HW1/wikitext-103/train-103.1.parquet')
wikitext_103_train_2 = pd.read_parquet('/content/drive/MyDrive/BUDT 758O:  Special Topics in Decision, Operations and Information Technologies; Designing AI Systems/HW1/wikitext-103/train-103.2.parquet')

In [ ]:
# Combine training datasets
wikitext_train = pd.concat([wikitext_2_train, wikitext_103_train_1,wikitext_103_train_2 ], ignore_index=True)

# Combine test datasets
wikitext_test = pd.concat([wikitext_103_test, wikitext_2_test], ignore_index=True)


In [ ]:
wikitext_train.head(10)
wikitext_test.head(10)

,text
0,
1,= Robert Boulter = \n
2,
3,"Robert Boulter is an English film , televisio..."
4,"In 2006 , Boulter starred alongside Whishaw i..."
5,
6,= = Career = = \n
7,
8,
9,= = = 2000 – 2005 = = = \n


In [ ]:
#Bigger datasets cause my PC to crash with the RAM reaching capacity (with GPU)
wikitext_train = wikitext_train.sample(frac=0.5, random_state=42)
wikitext_test = wikitext_test.sample(frac=0.5, random_state=42)


In [ ]:
len(wikitext_train)

367614

### Word Tokenization from the Text




In [ ]:
# Tokenize the text data
!pip install nltk
import nltk
nltk.download('punkt_tab')
def tokenize_text(text):
  tokens = word_tokenize(text)
  return tokens

wikitext_train['tokens'] = wikitext_train['text'].apply(tokenize_text)
wikitext_test['tokens'] = wikitext_test['text'].apply(tokenize_text)


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


### Vocabulary Building




In [ ]:
# Vocabulary
def build_vocab(data, min_freq=2):
    word_counts = Counter()
    for tokens in data:
        word_counts.update(tokens)

    vocabulary = [word for word, count in word_counts.items() if count >= min_freq]
    word_to_idx = {word: idx for idx, word in enumerate(vocabulary)}
    idx_to_word = {idx: word for idx, word in enumerate(vocabulary)}

    return word_to_idx, idx_to_word, vocabulary

word_to_idx, idx_to_word, vocabulary = build_vocab(wikitext_train['tokens'])
print(f"Vocabulary size: {len(vocabulary)}")


Vocabulary size: 163109


### Data loader
Process datasets to create a training loader and a testing loader. Do NOT use torchtext.dataset. You have to download the raw text and create a customized dataset and dataloader. (30 points).




In [ ]:
class Word2VecDataset(Dataset):
    def __init__(self, skipgrams, word_to_index):
        # Convert skipgrams to list of tuples (if not already done)
        self.skipgrams = [(item[0], item[1]) if isinstance(item, list) and len(item) >= 2 else (None, None) for item in skipgrams]
        self.word_to_index = word_to_index
    def __len__(self):
        return len(self.skipgrams)
    def __getitem__(self, idx):
        center_word, context_word = self.skipgrams[idx]
        # Center_word/context_word = not in vocabulary
        center_idx = self.word_to_index.get(center_word, 0)
        context_idx = self.word_to_index.get(context_word, 0)
        return torch.tensor(center_idx, dtype=torch.long), torch.tensor(context_idx, dtype=torch.long)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Create Word2VecDatasets
X_train_data = Word2VecDataset(wikitext_train['tokens'].values, word_to_idx)
X_test_data = Word2VecDataset(wikitext_test['tokens'].values, word_to_idx)


In [ ]:
#Dataloaders!
batch_size= 8
train_loader= DataLoader(X_train_data, batch_size= batch_size, shuffle= True)
test_loader= DataLoader(X_test_data, batch_size= batch_size, shuffle= True)

In [ ]:
#Shapes of the data loaders
for center_word, context_word in train_loader:
  print("Center word shape:", center_word.shape)
  print("Context word shape:", context_word.shape)
  break

Center word shape: torch.Size([8])
Context word shape: torch.Size([8])


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Word2Vec Model Architecture

(10 points)



In [ ]:
# Created Word2Vec Model
class Word2VecModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(Word2VecModel, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear_relu_stack= nn.Sequential(
        nn.Linear(embedding_dim, 256),
        nn.ReLU(),
        nn.Linear(256, vocab_size)
    )

    def forward(self, inputs):
        embeds = self.embeddings(inputs)
        out = self.linear_relu_stack(embeds)
        out = torch.log_softmax(out, dim=1)
        return out


In [ ]:
#Model Information
vocab_size = len(word_to_idx)
embedding_dim = 50
model = Word2VecModel(vocab_size, embedding_dim).to(device)
print(model)

Word2VecModel(
  (embeddings): Embedding(163109, 50)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=50, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=163109, bias=True)
  )
)


In [ ]:
# Print out the model parameters
for name, param in model.named_parameters():
  print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Layer: embeddings.weight | Size: torch.Size([163109, 50]) | Values : tensor([[ 0.8789,  1.1373,  0.5270, -0.4285,  0.4289,  0.0479, -0.9914, -0.1305,
         -0.1165,  0.7158, -0.6904,  1.1329,  0.2386, -0.8710, -0.7587, -0.2570,
          1.5321,  0.0482,  0.1804,  0.1225, -1.4095, -0.2854, -0.6523, -0.9199,
         -1.1173, -0.2799,  0.2131, -0.6555, -0.8660,  0.0169,  1.2450, -2.0797,
          2.5489,  0.1546, -0.4614, -1.0415, -0.3659,  0.0794,  1.1344,  1.1287,
          0.0662, -0.7941,  1.1861, -2.0341, -0.7048,  1.6437,  0.4512,  0.6830,
         -0.3890, -1.1220],
        [-1.0857,  1.5148,  1.9085,  0.9305,  0.8900, -1.1085,  1.1131,  0.4108,
          0.1604,  0.4829,  0.7649, -0.8642, -0.8105, -0.3880,  2.0196,  1.0024,
          1.0155, -0.7586, -0.5614,  0.0247,  1.0230, -0.9563,  0.2243,  0.0065,
          0.9933,  0.9988,  0.0058, -0.2001, -0.7067, -0.1273,  0.9814, -0.9706,
         -0.7298, -1.2019, -1.4806, -1.6013,  0.6560, -0.1122,  1.4661,  0.3207,
          0.

### Training
Training process using mini-batch strategy (30 points).

In [ ]:
learning_rate= 1e-4
epochs= 5 #can change
#Loss function (case sensitive from multiclass classification model)
loss_fn= nn.CrossEntropyLoss()
#Optimizer (gradient decsent)
#Adaptive learning rate with momementum ADAM
optimizer= torch.optim.Adam(model.parameters(), lr= learning_rate)

In [ ]:
for epoch in range(epochs):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = loss_fn(output, target)
        loss.backward()
        optimizer.step()
      #compute prediction error
        pred = model(data)
        loss= loss_fn(pred, target)
        if batch_idx % 1000 == 0:
            print(f"Epoch: {epoch+1}/{epochs}, Batch: {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}")


Epoch: 1/1, Batch: 0/45952, Loss: 11.9284
Epoch: 1/1, Batch: 1000/45952, Loss: 4.5446
Epoch: 1/1, Batch: 2000/45952, Loss: 3.3849
Epoch: 1/1, Batch: 3000/45952, Loss: 2.0295
Epoch: 1/1, Batch: 4000/45952, Loss: 3.5320
Epoch: 1/1, Batch: 5000/45952, Loss: 5.4752
Epoch: 1/1, Batch: 6000/45952, Loss: 2.4937
Epoch: 1/1, Batch: 7000/45952, Loss: 3.8904
Epoch: 1/1, Batch: 8000/45952, Loss: 5.4398
Epoch: 1/1, Batch: 9000/45952, Loss: 2.7807
Epoch: 1/1, Batch: 10000/45952, Loss: 3.2880
Epoch: 1/1, Batch: 11000/45952, Loss: 4.8579
Epoch: 1/1, Batch: 12000/45952, Loss: 4.5857
Epoch: 1/1, Batch: 13000/45952, Loss: 4.0798
Epoch: 1/1, Batch: 14000/45952, Loss: 3.6567
Epoch: 1/1, Batch: 15000/45952, Loss: 1.4593
Epoch: 1/1, Batch: 16000/45952, Loss: 2.5156
Epoch: 1/1, Batch: 17000/45952, Loss: 3.6534
Epoch: 1/1, Batch: 18000/45952, Loss: 4.2292
Epoch: 1/1, Batch: 19000/45952, Loss: 5.5023
Epoch: 1/1, Batch: 20000/45952, Loss: 2.2409
Epoch: 1/1, Batch: 21000/45952, Loss: 1.6318
Epoch: 1/1, Batch: 220

In [ ]:
# Evaluation
model.eval()
total_loss = 0
correct = 0
total = 0
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = loss_fn(output, target)  # Calculate loss for each batch
        total_loss += loss.item() * data.size(0) #add loss to total loss
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

    total_loss /= total #average across all samples
    accuracy = 100 * correct / total
    print(f"Test Loss: {total_loss:.4f} | Test Accuracy: {accuracy:.2f}%")

Test Loss: 3.7752 | Test Accuracy: 55.60%


### Inference

Print top K (e.g., K=10) similar words for any give word (the input) (10 points).

In [ ]:
#User gives input word
word_prompt = input("Enter the word prompt: ")
print(f"You entered: {word_prompt}")

Enter the word prompt: the
You entered: the


In [ ]:
#Inference: Print top K (e.g., K=10) similar words for input

import torch

def get_top_similar_words(word, model, word_to_idx, idx_to_word, k=10):
    try:
        word_idx = word_to_idx[word]
    except KeyError:
        print(f"Word '{word}' not found in vocabulary.")
        return []
    word_vector = model.embeddings.weight[word_idx]
    # Cosine similarity
    similarities = torch.cosine_similarity(word_vector.unsqueeze(0), model.embeddings.weight, dim=1)
    #Top k most similar words
    _, indices = torch.topk(similarities, k + 1)
    similar_words = []
    for idx in indices[1:]:
      similar_words.append((idx_to_word[idx.item()], similarities[idx].item()))

    return similar_words

# Word embeddings from the trained model
word_embeddings = model.embeddings.weight.cpu().detach().numpy()
input_word = word_prompt
top_similar = get_top_similar_words(input_word, model, word_to_idx, idx_to_word)

print(f"Top 10 similar words to '{input_word}':")
for word, similarity in top_similar:
    print(f"- {word}: {similarity:.4f}")


Top 10 similar words to 'the':
- cassava: 0.5752
- Geographic: 0.5585
- Yuhak: 0.5403
- garrulus: 0.5380
- Rendova: 0.5328
- driveway: 0.5296
- 252: 0.5277
- transcribe: 0.5136
- Ethos: 0.5123
- Harunobu: 0.5021


### Save Model Path in Drive

In [ ]:
torch.save(model.state_dict(),'/content/drive/MyDrive/BUDT 758O:  Special Topics in Decision, Operations and Information Technologies; Designing AI Systems/HW1/Word2Vec_model.pth')


##References:


1. GeeksforGeeks. (n.d.). Implement your own Word2Vec Skip-Gram model in Python. GeeksforGeeks. Retrieved February 12, 2025, from https://www.geeksforgeeks.org/implement-your-own-word2vecskip-gram-model-in-python/

2. Google. (n.d.). Gemini AI. Retrieved February 12, 2025, from https://deepmind. google/technologies/gemini/

3. Google DeepMind. (2023, December 6). Gemini: Our largest and most capable AI model [Video]. YouTube. https://www.youtube.com/watch?v=zjaRNfvNMTs